In [2]:
library(keras)
library(tensorflow)
library(tidyverse)
library(recipes)
library(randomForest)
library(dplyr) 
library(xgboost)
library(caret)

Warning message:
"le package 'keras' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tensorflow' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyverse' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'ggplot2' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tibble' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'readr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'forcats' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'lubridate' a été compilé avec la version R 4.2.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.2     ✔ tibble    3.2.1
✔ lubridate 1.9.2    

In [3]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

Let us try two strategies for the ensemble prediction. First, the most simple one would be to cast a majority vote among our 4 models.

As the 4 models perform at quite similar levels, even if some are slightly better than others, a type of voting could work. We will use a soft voting scheme, where the weighted sum of probabilities for each class is computed from the predictions of the models. 

After that, we will try to create an adaboosted version of our predictors. While this scheme is mostly used for a lot of weak learners, we will try here to use it with a few strong learners.

In [1]:
data<-read.csv("data_target_encoding.csv",stringsAsFactors = T)
test<-read.csv("test_target_encoding.csv",stringsAsFactors = T)


dataNN<-read.csv("data_target_encoding_NN.csv",stringsAsFactors = T)

In [4]:
classConverter <- function(predict_data,test_data) {
    yhat<-data.frame(matrix(0,ncol = 1, nrow = nrow(predict_data)))
    y<-data.frame(matrix(0,ncol = 1, nrow = nrow(predict_data)))
    for (i in 1:nrow(predict_data)){
        yhat[i,]<-which.max(predict_data[i,])
        y[i,]<-which.max(test_data[i,]) 
    }
    mylist <- list(yhat,y)
}

In [5]:
# drop = F is used to preserve the structure of the data as data.frame (see https://www.r-bloggers.com/2018/02/r-tip-use-drop-false-with-data-frames/)
set.seed(2) 

targets<-which(grepl('damage_grade',colnames(dataNN)))
n<-ncol(dataNN)
correlation<-abs(cor(dataNN[,-targets,drop=F],dataNN[,targets,drop=F]))
selected<-c()
candidates<-1:(n-length(targets))

    #mRMR ranks the variables by taking account not only the correlation with the output, but also by avoiding redudant variables
    for (j in 1:n) {
        redundancy_score<-numeric(length(candidates))
        
        if (length(selected)>0) {
            # Compute the correlation between the selected variables and the candidates on the training set
            cor_selected_candidates<-abs(cor(dataNN[,selected,drop=F],dataNN[,candidates,drop=F]))
            # Compute the mean correlation for each candidate variable, across the selected variables
            redundancy_score<-apply(cor_selected_candidates,2,mean)
        }
        
        # mRMR: minimum Redundancy Maximum Relevancy
        mRMR_score<-correlation[candidates]-redundancy_score
        
        # Select the candidate variable that maximises the mRMR score
        selected_current<-candidates[which.max(mRMR_score)]
        selected<-c(selected,selected_current)
        
        # Remove the selected variables from the candidates
        candidates<-setdiff(candidates,selected_current)
    }
    
    rankingNN <- selected
dataNN<-dataNN[,rankingNN[1:22]]


targets<-which(grepl('damage_grade',colnames(dataNN)))

normalizer<-layer_normalization(axis = -1L)  %>%  
adapt(as.matrix(dataNN[,-targets]))

neuralmodel <- keras_model_sequential() %>% 
normalizer  %>% 
layer_dense(37, activation = 'relu') %>%
layer_dense(3,activation='softmax')

neuralmodel %>% compile(
    loss = 'categorical_crossentropy',
    optimizer = optimizer_adam(0.0001),
    metrics=c('AUC')
  )

  data <- filter(data,age < 995)  #we remove outliers of age to have the same nb of rows for both

geo_level_3_mean_damage,foundation_type_i,age,has_superstructure_mud_mortar_stone,geo_level_3_sd_damage,geo_level_2_mean_damage,other_floor_type_q,has_superstructure_rc_engineered,foundation_type_r,count_families,⋯,geo_level_2_sd_damage,roof_type_x,other_floor_type_j,has_superstructure_rc_non_engineered,foundation_type_w,has_superstructure_cement_mortar_brick,has_secondary_use_rental,has_superstructure_other,ground_floor_type_f,legal_ownership_status_a
<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,⋯,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
-0.67033928,0,0.18658274,1,-1.59543141,-0.71177035,0,0,1,0,⋯,-1.75559843,0,1,0,0,0,0,0,1,0
1.90750849,0,-1.08848394,1,-2.58539146,2.06482679,1,0,1,1,⋯,-3.15856423,0,0,0,0,0,0,0,1,0
1.90750849,0,-0.83347061,1,-2.58539146,2.08531393,1,0,1,1,⋯,-3.40485914,0,0,0,0,0,0,0,1,0
-0.28545229,0,2.99172946,1,0.65675514,-0.82977394,1,0,1,1,⋯,-0.03013402,0,0,0,0,0,0,0,1,0
0.55458199,0,0.95162276,0,0.44930423,-0.05663109,1,0,1,1,⋯,0.08862635,0,0,0,0,0,0,0,1,0
-1.41437293,0,-1.08848394,0,0.64466428,-1.61496209,0,0,0,1,⋯,0.74577781,0,0,0,1,0,0,0,1,1
0.96766816,0,-1.08848394,1,0.78676486,1.27613792,0,0,1,1,⋯,0.04987942,0,0,0,0,0,0,0,1,0
0.09264440,0,0.44159608,1,0.50439096,-0.32741660,1,0,1,0,⋯,-0.99382566,0,0,0,0,0,0,0,1,0
-0.40594464,0,0.95162276,1,-1.01805261,-0.29608325,1,0,1,2,⋯,-1.22466166,0,0,0,0,0,0,0,1,0


In [6]:
set.seed(2)
n_trees <- 100
nfeat <- ncol(data)-1 #I remove 1 to remove damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)))
target_variable <- match('damage_grade', colnames(data))


k = 5
accuracy_vec <- data.frame(matrix(0,nrow=k,ncol=4))
colnames(accuracy_vec)<-c('RF','NN','XG','Softvote')


nrows<-nrow(data)

# 1. Shuffle the dataset randomly.
data_idx <- sample(1:nrows)

# 2. Split the dataset into k groups
max <- ceiling(nrows/k)
splits <- split(data_idx, ceiling(seq_along(data_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)

#3. For each unique group:
for (i in 1:k){
  #3.1 Take the group as a hold out or test data set
  test_data <- data[splits[[i]],]
  test_dataNN <- dataNN[splits[[i]],]
  #3.2 Take the remaining groups as a training data set
  train_data <- data[-splits[[i]],]   
  train_dataNN <- dataNN[-splits[[i]],]   

  model <- randomForest(x=train_data[,-c(target_variable)],
                      y=as.factor(train_data[,c(target_variable)]),
                      ntree=n_trees,mtry=m_tries,keep.forest=TRUE,importance=TRUE,type='prob')
  yhat_RF<-predict(model,test_data[,-c(target_variable)],type='prob')
  yhaty_RF <- classConverter(yhat_RF,test_dataNN)
  accuracy_vec[i,1]<-F1_Score_micro(yhaty_RF[[2]][,],yhaty_RF[[1]][,])


  model_history <- neuralmodel %>% fit(
  as.matrix(train_data[,-targets]),
  as.matrix(train_data[,targets]),
  validation_split = 0.2,
  verbose = 0,
  epochs = 30
  )
  yhat_NN <- predict(neuralmodel, as.matrix(test_data[-targets]))
  yhaty_NN <- classConverter(yhat_NN,test_data)
  accuracy_vec[i,2]<-F1_Score_micro(yhaty_NN[[2]][,],yhaty_NN[[1]][,])


  yhat_softvote <- (yhat_RF + yhat_NN + yhat_XG)/3
  yhat_classvote <- data.frame(matrix(0,nrow=nrow(yhat_softvote),ncol=1))
  for (j in 1:nrow(yhat_softvote)){
    yhat_classvote[j,]<- which.max(yhat_softvote[j,]) 
  }
  
  accuracy_vec[i,4] <- F1_Score_micro(yhat_classvote[,],as.factor(test_data[,c(target_variable)]))
  setTxtProgressBar(pb, i)
  print(accuracy_vec)

}

  |                                                                      |   0%

ERROR: Error in matrix(0, ncol = 1, nrow = nrow(predict_data)): plage non numérique pour une matrice
